# Preparación de datos para el análisis

Usaremos los datos de nacimientos de Barcelona de 2024. Cada fila representa un registro agregado, con un recuento en la columna `Valor`. Nuestro objetivo es preparar una tabla fiable y explicar qué podemos concluir a partir de ella.

Antes de empezar, revisa la [guía de entorno](../01_environment_pyspark.md). Trabajaremos con tablas de resultados.

## 1. Leer los datos y entender qué representa una fila

Una fila contiene un recuento y sus referencias geográficas. Contar filas responde cuántos registros tenemos. Sumar recuentos responde cuántos nacimientos aparecen en los valores publicados. El marcador `..` indica un valor ausente para nuestro análisis. Conservaremos esa diferencia al leer el archivo.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp

from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.master("local[2]").appName("Preparación de datos").getOrCreate()

archivo = "2024_pad_nai_mdbas.csv"
rutas = [Path("data") / archivo, Path("../data") / archivo, Path(archivo)]
ruta = next((opcion for opcion in rutas if opcion.is_file()), None)
if ruta is None:
    raise FileNotFoundError(f"Coloca {archivo} en data o súbelo a Colab.")

datos = spark.read.csv(str(ruta.resolve()), header=True, inferSchema=True, nullValue="..")
datos.select("Any", "Nom_Districte", "Nom_Barri", "Seccio_Censal", "Valor").show(5)

## 2. Hacer comprobaciones de calidad

Una prueba de calidad comprueba una regla que esperamos que cumplan los datos. Aquí esperamos que el año sea 2024 y que los recuentos conocidos sean enteros y no negativos. También contamos los valores ausentes y buscamos filas idénticas. Si aparecen filas repetidas, primero investigamos su origen.

Un valor ausente significa que desconocemos el recuento. Un cero es un recuento conocido.

In [ ]:
calidad = datos.agg(
    F.count("*").alias("registros"),
    F.count("Valor").alias("recuentos_conocidos"),
    F.count(F.when(F.col("Valor").isNull(), 1)).alias("recuentos_ausentes"),
    F.count(F.when(F.col("Any").isNull() | (F.col("Any") != 2024), 1)).alias("anios_invalidos"),
    F.count(F.when(
        F.col("Valor").isNotNull() & ~F.col("Valor").cast("string").rlike(r"^[0-9]+$"),
        1,
    )).alias("recuentos_invalidos"),
).first()

print(calidad.asDict())
print("Filas idénticas adicionales:", calidad["registros"] - datos.distinct().count())

assert calidad["registros"] > 0
assert calidad["anios_invalidos"] == calidad["recuentos_invalidos"] == 0
assert calidad["registros"] == calidad["recuentos_conocidos"] + calidad["recuentos_ausentes"]

## 3. Preparar y guardar la tabla en Parquet

Seleccionamos las columnas necesarias, usamos nombres claros y conservamos los valores ausentes. Guardamos la tabla preparada como Parquet y la volvemos a leer. Así podemos reutilizarla para el análisis sin repetir la preparación del CSV.

In [ ]:
preparados = datos.select(
    F.col("Any").alias("anio"),
    F.col("Nom_Districte").alias("distrito"),
    F.col("Nom_Barri").alias("barrio"),
    F.col("Seccio_Censal").alias("seccion_censal"),
    F.col("Valor").cast("long").alias("nacimientos"),
)

ruta_parquet = Path(mkdtemp(prefix="nacimientos_")) / "nacimientos.parquet"
preparados.write.parquet(str(ruta_parquet))
analisis = spark.read.parquet(str(ruta_parquet))
analisis.printSchema()
assert analisis.count() == calidad["registros"]

## 4. Resumir e interpretar

El mínimo y el máximo muestran los extremos de los recuentos conocidos por registro. La media reparte el total conocido entre esos registros. La mediana aproximada indica el valor central y ayuda a describir un registro típico cuando hay valores altos o bajos. La desviación estándar indica cuánto varían los recuentos alrededor de la media.

Estas cifras describen **recuentos por registro**. Para comparar tasas de natalidad entre distritos necesitaríamos también un denominador de población adecuado.

In [ ]:
analisis.agg(
    F.count("nacimientos").alias("conocidos"),
    F.min("nacimientos").alias("minimo"),
    F.max("nacimientos").alias("maximo"),
    F.round(F.avg("nacimientos"), 2).alias("media"),
    F.percentile_approx("nacimientos", 0.5).alias("mediana_aprox"),
    F.round(F.stddev_pop("nacimientos"), 2).alias("desviacion"),
).show()

## 5. Comparar distritos con contexto

Ordena los distritos por nacimientos conocidos. Antes de interpretar el orden, mira cuántos registros se publicaron con recuento y cuántos quedaron sin valor. El total de nacimientos conocidos puede ser incompleto. El porcentaje de registros con valor mide la cobertura de los registros, no el porcentaje de nacimientos publicados.

In [ ]:
por_distrito = analisis.groupBy("distrito").agg(
    F.count("*").alias("registros"),
    F.count("nacimientos").alias("con_valor"),
    F.sum("nacimientos").alias("nacimientos_conocidos"),
).withColumn(
    "sin_valor", F.col("registros") - F.col("con_valor"),
).withColumn(
    "cobertura_pct", F.round(100 * F.col("con_valor") / F.col("registros"), 1),
)

por_distrito.orderBy(F.desc_nulls_last("nacimientos_conocidos")).show(truncate=False)

## 6. Práctica y puesta en común

1. Haz el mismo resumen por distrito y barrio. Incluye el número de registros, los recuentos conocidos y el mínimo y máximo de nacimientos por registro.
2. Comprueba que la suma del número de registros de todos los grupos coincide con el número de filas de la tabla preparada.
3. Elige un barrio y redacta dos frases. En la primera, explica un resultado citando una cifra y su unidad. En la segunda, explica qué información adicional necesitarías para interpretar ese resultado.

Como orientación, pregunta si los extremos son recuentos por registro o totales del barrio. Revisa también cuántos valores faltan antes de comparar barrios.

In [ ]:
# Escribe aquí tu solución.

In [ ]:
spark.stop()

## Navegación

[Volver al README](../README.md) · [Siguiente: Visualización de datos](09_data_visualization.ipynb)